In [1]:
import pandas as pd
import json
import ir_datasets
from src.data import DATA_DIR_PROCESSED, DATA_DIR_RAW
import os
from topic_gen.evaluate import QrelsEvaluator, CohenKappa, MeanAverageError, AreaUnderReceiver, ROSKendallTau, ROSRBO, binarize_qrels, load_runs_from_path
import ir_measures
from src.data import parse_file_names
from pathlib import Path

from topic_gen import logger
logger.setLevel("DEBUG")

In [2]:
# Load generated qrels from path
BASE_DIR = DATA_DIR_PROCESSED / "qrels"

predictions = []
names = []
metadata_records = []
for result in os.listdir(BASE_DIR):
    names.append(result)

    # metadata
    with open(os.path.join(BASE_DIR, result, "metadata.json")) as f:
        metadata = json.load(f)
    metadata_records.append(metadata)

    # predictions
    qrels = binarize_qrels(ir_measures.read_trec_qrels(
        os.path.join(BASE_DIR, result, "qrels.csv.gz")))
    predictions.append(qrels)

In [3]:
# Evaluate qrels
res = QrelsEvaluator.experiment(
    predictions=predictions,
    references=binarize_qrels(ir_datasets.load(
        "disks45/nocr/trec-robust-2004").qrels_iter()),
    measures=[CohenKappa(), MeanAverageError(), AreaUnderReceiver()],
    bootstrap=20,
    names=names)

[topic_gen] [INFO] (evaluate.py:249) Qrels in reference but not in predictions: 12 / 2939
[topic_gen] [INFO] (evaluate.py:249) Qrels in reference but not in predictions: 8 / 2943
[topic_gen] [INFO] (evaluate.py:249) Qrels in reference but not in predictions: 0 / 2951
[topic_gen] [INFO] (evaluate.py:249) Qrels in reference but not in predictions: 11 / 2940
[topic_gen] [INFO] (evaluate.py:249) Qrels in reference but not in predictions: 4 / 2947
[topic_gen] [INFO] (evaluate.py:249) Qrels in reference but not in predictions: 10 / 2941
[topic_gen] [INFO] (evaluate.py:249) Qrels in reference but not in predictions: 0 / 2951
[topic_gen] [INFO] (evaluate.py:249) Qrels in reference but not in predictions: 0 / 2951
[topic_gen] [INFO] (evaluate.py:249) Qrels in reference but not in predictions: 2 / 2949
[topic_gen] [INFO] (evaluate.py:249) Qrels in reference but not in predictions: 15 / 2936
[topic_gen] [INFO] (evaluate.py:249) Qrels in reference but not in predictions: 9 / 2942
[topic_gen] [INFO

In [4]:
# metadata table
metadata = pd.DataFrame(metadata_records)
metadata = metadata.join(pd.json_normalize(
    metadata["topics"]).add_prefix("topics_"))
metadata.drop(columns=["topics"], inplace=True)
metadata["topics_prompt"] = metadata["topics_prompt"].apply(
    lambda p: str(Path(p).stem) if pd.notnull(p) else "human")
metadata["prompt"] = metadata["prompt"].apply(lambda p: str(Path(p).stem))
metadata["model"] = metadata["model"].str.replace("-MT1000", "")
metadata["model"] = metadata["model"].str.replace("-MT100", "")

In [5]:
def format_score(row):
    return f"{row['value']:.2f} ± {row['ci']:.2f}"


table = pd.DataFrame(res)
table["score"] = table.apply(format_score, axis=1)
table = table.pivot(index="name", columns="measure",
                    values="score").reset_index()

In [6]:
table = table.merge(metadata, left_on="name", right_on="date")

## Alignment
RQ: How well align qrels based on generated topics with qrels based on the original qrels?
 

### Prompting Strategies
**Important:** the columns `ndocpos` and `ndocsneg` state values even if they are ignored by the prompts.


Prompts:
- Trec: query variants and relevant docs
- trec-query: only query
- trec-contrastive: query, relevant, and irelevant documents
  

### Findings
- Judgments based on the original topics are always better. Sometimes just a little!
- Some settings show a substantial drop in alignment, for example, for `qwen3-30B` on generated topics with the `trec` prompt with one query variant and one relevant document, the Cohen's $\kappa$ agreement to human labels drops from the substantial agreement of 0.75 to a moderate agreement of 0.52.
- More context always helps.
- Contrastive prompting is the best.

In [7]:
prompt_sorter = ["human", "trec", "trec-query", "trec-contrastive"]
table["topics_prompt"] = pd.Categorical(
    table["topics_prompt"], prompt_sorter)

table[(table["prompt"] == "-DNA-zero-shot") & ((table["model"] == table["topics_model"]) | (table["topics_model"] == "trec assessors"))]\
    .sort_values(by=["model", "topics_prompt"], ascending=[True, True])[["topics_prompt", "model", "topics_nqueries", "topics_ndocspos", "topics_ndocsneg", "CohenKappa", "MeanAverageError", "AreaUnderReceiver"]]

,topics_prompt,model,topics_nqueries,topics_ndocspos,topics_ndocsneg,CohenKappa,MeanAverageError,AreaUnderReceiver
46,human,gpt-4.1,NaN,NaN,NaN,0.64 ± 0.03,0.17 ± 0.01,0.81 ± 0.01
53,human,gpt-oss-120B,NaN,NaN,NaN,0.70 ± 0.02,0.14 ± 0.01,0.84 ± 0.01
80,human,gpt-oss-120B,NaN,NaN,NaN,0.69 ± 0.02,0.15 ± 0.01,0.84 ± 0.01
72,trec,gpt-oss-120B,1.0,1.0,3.0,0.42 ± 0.02,0.32 ± 0.01,0.74 ± 0.01
73,trec,gpt-oss-120B,3.0,2.0,3.0,0.46 ± 0.02,0.29 ± 0.01,0.76 ± 0.01
74,trec,gpt-oss-120B,5.0,3.0,3.0,0.47 ± 0.02,0.28 ± 0.01,0.76 ± 0.01
75,trec-query,gpt-oss-120B,1.0,3.0,3.0,0.33 ± 0.02,0.38 ± 0.01,0.72 ± 0.01
76,trec-query,gpt-oss-120B,3.0,3.0,3.0,0.33 ± 0.02,0.38 ± 0.01,0.72 ± 0.01
77,trec-query,gpt-oss-120B,5.0,3.0,3.0,0.40 ± 0.02,0.33 ± 0.01,0.74 ± 0.01
78,trec-contrastive,gpt-oss-120B,1.0,1.0,1.0,0.51 ± 0.02,0.26 ± 0.02,0.77 ± 0.01
